# Post-training an LLM for reasoning with GRPO in TRL

This notebook implements a minimal GRPO training loop similar to the tutorial you shared:
- Dataset: `AI-MO/NuminaMath-TIR`
- Base model: `Qwen/Qwen2-0.5B-Instruct`
- LoRA via PEFT
- Two rewards: format + accuracy (`math_verify`)

> Notes
- Training GRPO is GPU-heavy. For quick smoke tests, this notebook trains on small slices.
- Pushing to the Hub is optional and disabled by default.


## 1) Install dependencies

In [2]:
!pip install -U -q trl peft math_verify datasets accelerate transformers

# (Optional) If you want TRL's vLLM integration later:
# !pip install -U -q "trl[vllm]"

## 2) (Optional) Login to Hugging Face
Only needed if you want to `push_to_hub=True` later.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 3) Load dataset

In [24]:
from datasets import load_dataset

dataset_id = "AI-MO/NuminaMath-TIR"

# Small slices for a quick run; increase as you have compute
train_dataset, test_dataset = load_dataset(
    dataset_id,
    split=["train[:5%]", "test[:5%]"]
)

print("Train:", train_dataset)
print("Test:", test_dataset)
print("\nTrain columns:", train_dataset.column_names)

# Print a compact example (solutions can be long)
ex = train_dataset[0]
print("\nExample row (truncated):")
print({
    "problem": ex["problem"][:200] + ("..." if len(ex["problem"]) > 200 else ""),
    "solution": ex["solution"][:200] + ("..." if len(ex["solution"]) > 200 else ""),
    "messages_len": len(ex["messages"]) if "messages" in ex else None,
})


Train: Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 3622
})
Test: Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 5
})

Train columns: ['problem', 'solution', 'messages']

Example row (truncated):
{'problem': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'solution': 'To determine the coefficient of \\(x^2y^6\\) in the expansion of \\(\\left(\\frac{3}{5}x - \\frac{y}{2}\\right)^8\\), we can use the binomial theorem.\n\nThe binomial theorem states:\n\\[\n(a + b)^n = \\sum_{k=0}^{...', 'messages_len': 2}


## 4) Create conversation prompts (system + user)
We follow the DeepSeek-style format prompt using `<think>` and `<answer>` tags.

In [ ]:


def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ]
    }


Map:   0%|          | 0/3622 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['solution', 'prompt'],
    num_rows: 3622
})

Prompt example:
 [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> and </think> tags, and <answer> and </answer> tags, respectively.', 'role': 'system'}, {'content': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'role': 'user'}]


## 5) Load base model + tokenizer

In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [29]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
    "The reasoning process and answer are enclosed within <think> and </think> tags, and <answer> and </answer> tags, respectively."
)

def add_prompt(example, max_prompt_tokens=128):
    # Build a string prompt from raw dataset fields
    text = (
        "system\n" + SYSTEM_PROMPT + "\n"
        "user\n" + example["problem"] + "\n"
        "assistant\n"
    )

    # Optional truncation by tokens (recommended to control memory)
    ids = tokenizer(text, truncation=True, max_length=max_prompt_tokens)["input_ids"]
    text_trunc = tokenizer.decode(ids, skip_special_tokens=True)

    return {"prompt": text_trunc}

train_dataset = train_dataset.map(add_prompt, fn_kwargs={"max_prompt_tokens": 128})
test_dataset  = test_dataset.map(add_prompt, fn_kwargs={"max_prompt_tokens": 128})

# Keep only what GRPOTrainer + reward needs
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in ["prompt", "solution"]])
test_dataset  = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["prompt", "solution"]])

print("COLUMNS:", train_dataset.column_names)
print("Prompt example:\n", train_dataset[0]["prompt"][:400])


Map:   0%|          | 0/3622 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

COLUMNS: ['solution', 'prompt']
Prompt example:
 system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> and </think> tags, and <answer> and </answer> tags, respectively.
user
What is the coefficient of $x^2y^6$ in the expansion


## 6) Configure LoRA (PEFT)

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 7) Reward functions
- **Format reward**: checks the presence of `<think>...</think>` and `<answer>...</answer>`
- **Accuracy reward**: uses `math_verify` to compare extracted answers vs ground truth

In [7]:
import re
from math_verify import LatexExtractionConfig, parse, verify

FORMAT_PATTERN = re.compile(r"(?s).*<think>.*?</think>.*<answer>.*?</answer>.*")

def format_reward(completions, **kwargs):
    # TRL provides `completions` as a list of list-of-messages; we take the first message content.
    texts = [c[0]["content"] for c in completions]
    return [1.0 if FORMAT_PATTERN.match(t) else 0.0 for t in texts]

def accuracy_reward(completions, **kwargs):
    solutions = kwargs["solution"]
    texts = [c[0]["content"] for c in completions]
    rewards = []
    for text, sol in zip(texts, solutions):
        gold = parse(sol, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        pred = parse(text, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        if len(gold) != 0:
            try:
                rewards.append(float(verify(pred, gold)))
            except Exception:
                rewards.append(0.0)
        else:
            rewards.append(1.0)
    return rewards

print("Reward fns ready")

Reward fns ready


## 8) Configure GRPO + train

In [30]:
from trl import GRPOConfig, GRPOTrainer
import wandb

training_args = GRPOConfig(
    output_dir="Qwen2-0.5B-GRPO-test",
    learning_rate=1e-5,
    remove_unused_columns=False,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    bf16=True,

    max_completion_length=64,
    num_generations=4,

    report_to=["wandb"],     # ✅ logs to W&B instead
    run_name="qwen2-grpo",
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    push_to_hub=False,
)

# (Optional) vLLM integration (requires `pip install "trl[vllm]"`)
# training_args.use_vllm = True
# training_args.vllm_mode = "colocate"   # or "server" with a separate vLLM server

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,  # ✅ TRL 0.28.0
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
)


trainer.train()
trainer.save_model(training_args.output_dir)

print("Saved to:", training_args.output_dir)

Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


NameError: name 'wandb' is not defined

## 9) Quick evaluation on a test sample

In [ ]:
import time

def build_prompt_text(prompt_messages):
    # Simple concat; you can also use tokenizer.apply_chat_template if your tokenizer supports it.
    return "\n".join([f"{m['role'].upper()}: {m['content']}" for m in prompt_messages]) + "\nASSISTANT:"

def generate_with_reasoning(prompt_messages, max_new_tokens=256):
    text = build_prompt_text(prompt_messages)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    end = time.time()

    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    num_input_tokens = inputs["input_ids"].shape[1]
    num_generated_tokens = out.shape[1] - num_input_tokens
    return generated, (end - start), int(num_generated_tokens)

sample = test_dataset[0]
print("Prompt messages:\n", sample["prompt"])

gen_text, dt, n_tok = generate_with_reasoning(sample["prompt"], max_new_tokens=256)
print("\n--- Generated ---\n")
print(gen_text)
print(f"\nInference time: {dt:.2f}s | Generated tokens: {n_tok}")

## 10) (Optional) Push to Hub
Uncomment if you logged in and want to upload.

In [ ]:
# trainer.push_to_hub(dataset_name=dataset_id)